# Staging: Prep

## Setup

### Imports

In [1]:
from pathlib import Path

import duckdb

### Configuration

In [2]:
# Locate the project root (folder containing requirements.txt), independent of cwd
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()
)

data_vault_path = str(PROJECT_ROOT / "data" / "warehouse" / "data_vault.duckdb")

### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
    CREATE SCHEMA IF NOT EXISTS prep;
""")

## Prepare Tables

### Customer Table

In [5]:
con.execute("""
CREATE OR REPLACE TABLE prep.customers AS
SELECT
    -- Business Keys / IDs
    TRIM(customer_id) AS customer_id,
    TRIM(customer_unique_id) AS customer_unique_id,

    -- Descriptive Attributes
    TRIM(CAST(customer_zip_code_prefix AS VARCHAR))
        AS customer_zip_code_prefix,

    LOWER(TRIM(customer_city)) AS customer_city,
    UPPER(TRIM(customer_state)) AS customer_state,

    -- Hash Key
    MD5(TRIM(customer_id)) AS customer_hk,

    -- HashDiff for change detection
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(TRIM(customer_unique_id), ''),
            COALESCE(TRIM(CAST(customer_zip_code_prefix AS VARCHAR)), ''),
            COALESCE(LOWER(TRIM(customer_city)), ''),
            COALESCE(UPPER(TRIM(customer_state)), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'customers' AS record_source

FROM staging.raw_customers;
""")

In [6]:
con.execute(""" select * from prep.customers""").df().head(5)

C:\Users\JacksonDivakarRajasi\AppData\Local\Temp\ipykernel_34344\3754097760.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  con.execute(""" select * from prep.customers""").df().head(5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_hk,hash_diff,load_date,record_source
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,b2e02fc25238d1dfc1de0ecdd902397d,65cea53690a671b8004195b963c8b9d1,2026-11-05 14:14:43.974584+05:30,customers
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,a7a275ec3b2e1646e730fdbef5f84cd4,8e5780036419a1ba755c2d63767c0c24,2026-11-05 14:14:43.974584+05:30,customers
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,3512e4ca5527f2f699b4ff1272854d8c,5d75aa9702e9bb0f9d2757a6e25f2eed,2026-11-05 14:14:43.974584+05:30,customers
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,6030e0796931d09ee56a989f520d4e4f,a65b3ae008f5215f3505a9cc4e3712bc,2026-11-05 14:14:43.974584+05:30,customers
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,7c516732a6d8b996750ac2825acbfa09,ac9f91c8393ebfab067bd0f8f2316efc,2026-11-05 14:14:43.974584+05:30,customers


### Order Payments Table

In [7]:
con.execute("""
CREATE OR REPLACE TABLE prep.order_payments AS
SELECT
    -- Order Business Key
    TRIM(order_id) AS order_id,

    -- Payment occurrence within the order
    CAST(payment_sequential AS INTEGER) AS payment_sequential,

    -- Payment attributes
    LOWER(TRIM(payment_type)) AS payment_type,
    CAST(payment_installments AS INTEGER) AS payment_installments,
    CAST(payment_value AS DECIMAL(18,2)) AS payment_value,

    -- Order Hub Hash Key
    MD5(TRIM(order_id)) AS order_hk,

    -- Order-Payment occurrence / Link Hash Key
    MD5(
        CONCAT_WS(
            '||',
            TRIM(order_id),
            CAST(payment_sequential AS VARCHAR)
        )
    ) AS order_payment_hk,

    -- Link Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(LOWER(TRIM(payment_type)), ''),
            COALESCE(CAST(payment_installments AS VARCHAR), ''),
            COALESCE(CAST(payment_value AS VARCHAR), '')
        )
    ) AS hash_diff,

    CURRENT_TIMESTAMP AS load_date,
    'order_payments' AS record_source

FROM staging.raw_order_payments;
""")

### Order Reviews

In [8]:
con.execute("""
CREATE OR REPLACE TABLE prep.order_reviews AS
SELECT
    -- Business Keys
    TRIM(review_id) AS review_id,
    TRIM(order_id) AS order_id,

    -- Review Attributes
    CAST(review_score AS INTEGER) AS review_score,
    TRIM(review_comment_title) AS review_comment_title,
    TRIM(review_comment_message) AS review_comment_message,

    CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,

    -- Hub Hash Keys
    MD5(TRIM(review_id)) AS review_hk,
    MD5(TRIM(order_id)) AS order_hk,

    -- Link Hash Key: Order <-> Review
    MD5(
        CONCAT_WS(
            '||',
            TRIM(order_id),
            TRIM(review_id)
        )
    ) AS order_review_hk,

    -- Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(CAST(review_score AS VARCHAR), ''),
            COALESCE(TRIM(review_comment_title), ''),
            COALESCE(TRIM(review_comment_message), ''),
            COALESCE(CAST(review_creation_date AS VARCHAR), ''),
            COALESCE(CAST(review_answer_timestamp AS VARCHAR), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'order_reviews' AS record_source

FROM staging.raw_order_reviews;
""")

### Geolocation Reference Table

In [9]:
con.execute("""
CREATE OR REPLACE TABLE prep.geolocation AS
SELECT
    TRIM(CAST(geolocation_zip_code_prefix AS VARCHAR))
        AS geolocation_zip_code_prefix,

    AVG(CAST(geolocation_lat AS DOUBLE))
        AS geolocation_lat,

    AVG(CAST(geolocation_lng AS DOUBLE))
        AS geolocation_lng,

    LOWER(TRIM(MIN(geolocation_city)))
        AS geolocation_city,

    UPPER(TRIM(MIN(geolocation_state)))
        AS geolocation_state,

    CURRENT_TIMESTAMP AS load_date,
    'geolocation' AS record_source

FROM staging.raw_geolocation

GROUP BY
    TRIM(CAST(geolocation_zip_code_prefix AS VARCHAR));
""")

### Product Category Mapping

In [10]:
con.execute("""
CREATE OR REPLACE TABLE prep.product_category_translation AS
SELECT
    LOWER(TRIM(product_category_name))
        AS product_category_name,

    LOWER(TRIM(product_category_name_english))
        AS product_category_name_english,

    CURRENT_TIMESTAMP AS load_date,
    'product_category_translation' AS record_source

FROM staging.raw_product_category_name_translation;
""")

### Product Table

In [11]:
con.execute("""
CREATE OR REPLACE TABLE prep.products AS
SELECT
    -- Business Key
    TRIM(product_id) AS product_id,

    -- Product Attributes
    LOWER(TRIM(product_category_name)) AS product_category_name,

    CAST(product_name_lenght AS INTEGER) AS product_name_length,
    CAST(product_description_lenght AS INTEGER) AS product_description_length,
    CAST(product_photos_qty AS INTEGER) AS product_photos_qty,

    CAST(product_weight_g AS INTEGER) AS product_weight_g,
    CAST(product_length_cm AS INTEGER) AS product_length_cm,
    CAST(product_height_cm AS INTEGER) AS product_height_cm,
    CAST(product_width_cm AS INTEGER) AS product_width_cm,

    -- Product Hub Hash Key
    MD5(TRIM(product_id)) AS product_hk,

    -- Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(LOWER(TRIM(product_category_name)), ''),
            COALESCE(CAST(product_name_lenght AS VARCHAR), ''),
            COALESCE(CAST(product_description_lenght AS VARCHAR), ''),
            COALESCE(CAST(product_photos_qty AS VARCHAR), ''),
            COALESCE(CAST(product_weight_g AS VARCHAR), ''),
            COALESCE(CAST(product_length_cm AS VARCHAR), ''),
            COALESCE(CAST(product_height_cm AS VARCHAR), ''),
            COALESCE(CAST(product_width_cm AS VARCHAR), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'products' AS record_source

FROM staging.raw_products;
""")

### Sellers Table

In [12]:
con.execute("""
CREATE OR REPLACE TABLE prep.sellers AS
SELECT
    -- Business Key
    TRIM(seller_id) AS seller_id,

    -- Seller Attributes
    TRIM(CAST(seller_zip_code_prefix AS VARCHAR))
        AS seller_zip_code_prefix,

    LOWER(TRIM(seller_city)) AS seller_city,
    UPPER(TRIM(seller_state)) AS seller_state,

    -- Seller Hub Hash Key
    MD5(TRIM(seller_id)) AS seller_hk,

    -- Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(TRIM(CAST(seller_zip_code_prefix AS VARCHAR)), ''),
            COALESCE(LOWER(TRIM(seller_city)), ''),
            COALESCE(UPPER(TRIM(seller_state)), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'sellers' AS record_source

FROM staging.raw_sellers;
""")

### Order Table

In [13]:
con.execute("""
CREATE OR REPLACE TABLE prep.orders AS
SELECT
    -- Business Keys
    TRIM(order_id) AS order_id,
    TRIM(customer_id) AS customer_id,

    -- Order Attributes
    LOWER(TRIM(order_status)) AS order_status,

    CAST(order_purchase_timestamp AS TIMESTAMP)
        AS order_purchase_timestamp,

    CAST(order_approved_at AS TIMESTAMP)
        AS order_approved_at,

    CAST(order_delivered_carrier_date AS TIMESTAMP)
        AS order_delivered_carrier_date,

    CAST(order_delivered_customer_date AS TIMESTAMP)
        AS order_delivered_customer_date,

    CAST(order_estimated_delivery_date AS TIMESTAMP)
        AS order_estimated_delivery_date,

    -- Hub Hash Keys
    MD5(TRIM(order_id)) AS order_hk,
    MD5(TRIM(customer_id)) AS customer_hk,

    -- Order-Customer Link Hash Key
    MD5(
        CONCAT_WS(
            '||',
            TRIM(order_id),
            TRIM(customer_id)
        )
    ) AS order_customer_hk,

    -- Order Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(LOWER(TRIM(order_status)), ''),
            COALESCE(CAST(order_purchase_timestamp AS VARCHAR), ''),
            COALESCE(CAST(order_approved_at AS VARCHAR), ''),
            COALESCE(CAST(order_delivered_carrier_date AS VARCHAR), ''),
            COALESCE(CAST(order_delivered_customer_date AS VARCHAR), ''),
            COALESCE(CAST(order_estimated_delivery_date AS VARCHAR), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'orders' AS record_source

FROM staging.raw_orders;
""")

### Order Items

In [14]:
con.execute("""
CREATE OR REPLACE TABLE prep.order_items AS
SELECT
    -- Source Keys
    TRIM(order_id) AS order_id,
    CAST(order_item_id AS INTEGER) AS order_item_id,
    TRIM(product_id) AS product_id,
    TRIM(seller_id) AS seller_id,

    -- Transaction Attributes
    STRPTIME(
        shipping_limit_date,
        '%d-%m-%Y %H:%M'
    ) AS shipping_limit_date,

    CAST(price AS DECIMAL(18,2)) AS price,
    CAST(freight_value AS DECIMAL(18,2)) AS freight_value,

    -- Hub Hash Keys
    MD5(TRIM(order_id)) AS order_hk,
    MD5(TRIM(product_id)) AS product_hk,
    MD5(TRIM(seller_id)) AS seller_hk,

    -- Link Hash Key
    MD5(
        CONCAT_WS(
            '||',
            TRIM(order_id),
            CAST(order_item_id AS VARCHAR),
            TRIM(product_id),
            TRIM(seller_id)
        )
    ) AS order_item_hk,

    -- Link-Satellite HashDiff
    MD5(
        CONCAT_WS(
            '||',
            COALESCE(
                STRFTIME(
                    STRPTIME(shipping_limit_date, '%d-%m-%Y %H:%M'),
                    '%Y-%m-%d %H:%M:%S'
                ),
                ''
            ),
            COALESCE(CAST(price AS VARCHAR), ''),
            COALESCE(CAST(freight_value AS VARCHAR), '')
        )
    ) AS hash_diff,

    -- Metadata
    CURRENT_TIMESTAMP AS load_date,
    'order_items' AS record_source

FROM staging.raw_order_items;
""")

### Prep Tables (Summary)

In [15]:
con.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'prep'
ORDER BY table_name;
""").df()

,table_name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_translation
7,products
8,sellers
